# Tahap 0: Preprocessing Tambahan (Physical Renaming)
Kode opsional ini secara fisik mengganti nama file di local disk Anda agar skala keparahannya berurutan menjadi skala **1 sampai 6** (1 = Paling Ringan, 6 = Paling Parah), mengatasi gap pada data asli (misal data asli Reversal bernilai 1 digabung menjadi skor terparah 6, dan skor asli 9 menjadi skor 1).
Gunakan opsi ini jika Anda lebih mantap melihat datanya tertata secara rasional di File Explorer secara fisik!

In [11]:
import os
import time

# 1. Tentukan Path! 
root_dir = r'Gambo'

print("Mempersiapkan penggantian nama (Physical Renaming) skala 1-6...")

# Peta Inversi Skala Keparahan (Menghindari jarak kosong dan membalik hierarki)
score_map = {
    9: 1, # Asli 9 (Paling ringan) -> Jadi Skor AI 1
    8: 2,
    7: 3,
    6: 4,
    5: 5,
    4: 6, # Asli 4 (Corrected Paling Parah) -> Jadi Skor AI 6
    1: 6  # Asli 1 (Reversal) -> Sama-sama disatukan di puncak Skor AI 6 
}

to_rename = []
for root, dirs, files in os.walk(root_dir):
    if not ('Corrected' in root or 'Reversal' in root):
        continue
        
    for file in files:
        if file.endswith('.png'):
            prefix, separator = None, None
            if '_' in file:
                head = file.split('_')[0]
                if head.isdigit() and int(head) in score_map:
                    prefix, separator = int(head), '_'
            elif '-' in file:
                head = file.split('-')[0]
                if head.isdigit() and int(head) in score_map:
                    prefix, separator = int(head), '-'
            
            if prefix is not None:
                new_prefix = score_map[prefix] # Murni pembalikan skala via Peta (Mapping)!
                sisa_nama = file.split(separator, 1)[1]
                new_name = f"{new_prefix}{separator}{sisa_nama}"
                
                old_path = os.path.join(root, file)
                new_path = os.path.join(root, new_name)
                
                to_rename.append((old_path, new_path))

print(f"Total file yang akan dimapping ulang namanya: {len(to_rename)} file.")

# FASE 1: Rename ke nama SEMENTARA (TEMP) menghindari bentrok
temp_rename = []
for old, new in to_rename:
    temp_path = new + ".TEMP"
    os.rename(old, temp_path)
    temp_rename.append((temp_path, new))

# FASE 2: Hapus tulisan TEMP dengan toleransi bentrok Windows (Retry System)
for temp, final_new in temp_rename:
    berhasil = False
    for percobaan in range(10): # AI akan mencoba menggedor hak akses file maksimal 10x
        try:
            os.replace(temp, final_new) # Gunakan replace agar lebih tangguh
            berhasil = True
            break
        except PermissionError:
            time.sleep(0.1) # Beri jeda 0.1 detik agar Windows Defender/Thumbnail selesai membaca
            
    if not berhasil:
        print(f"Gagal memproses karena file dikunci permanen oleh Windows: {final_new}")

print("Selesai! Seluruh nama fisik file berhasil dipetakan ke skala konsisten (1 Ringan -> 6 Parah)!")

Mempersiapkan penggantian nama (Physical Renaming) skala 1-6...
Total file yang akan dimapping ulang namanya: 121835 file.
Selesai! Seluruh nama fisik file berhasil dipetakan ke skala konsisten (1 Ringan -> 6 Parah)!


# Tahap 1: Persiapan Dataset & Data Cleaning (Membuat CSV)
Karena kita menggunakan Google Colab nantinya, kita **tidak boleh** memindah-mindahkan 200.000 file gambar secara manual (karena proses I/O Google Drive sangat lambat).

Kode di bawah ini akan **mem-filter data kotor** (seperti file *NormalXXXX.png* yang tersesat di folder *Corrected*) dan menyimpan daftar lokasi file yang benar/rapi ke dalam sebuah file `master_dataset.csv`.

> **INGAT:** Anda harus mengubah tulisan lambang `root_dir` di bawah dengan path folder dataset Gambo asli Anda ketika sudah di-upload ke Google Drive/Colab!

In [12]:
import os
import pandas as pd
from pathlib import Path

root_dir = r'Gambo'

print("Membaca seluruh direktori...")
data = []

for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.endswith('.png'):
            parts = Path(root).parts
            try:
                split_type = parts[-2] # Mendapatkan nama folder: Train atau Test
                category = parts[-1]   # Mendapatkan nama folder: Normal, Corrected, Reversal
            except:
                continue
                
            path_full = os.path.join(root, file)
            data.append({
                'image_path': path_full,
                'file_name': file,
                'split': split_type,
                'folder_category': category
            })

df = pd.DataFrame(data)
print(f"Total gambar berserakan yang ditemukan: {len(df)} file.\n")

def get_score(row):
    filename = row['file_name']
    folder = row['folder_category']
    
    if 'Normal' in filename and folder != 'Normal':
        return 'DROP' 
        
    if folder == 'Normal':
        return 0 
        
    if folder in ['Corrected', 'Reversal']:
        # Karena nama file SUDAH DI-RENAME SECARA FISIK di tahap 0 menjadi skala 1-6
        # Maka CSV Cukup mengekstrak angkanya mentah-mentah saja tanpa pembalikan lagi
        if '_' in filename:
            prefix = filename.split('_')[0]
            if prefix.isdigit() and 1 <= int(prefix) <= 6: return int(prefix)
        if '-' in filename:
            prefix = filename.split('-')[0]
            if prefix.isdigit() and 1 <= int(prefix) <= 6: return int(prefix)
            
    return 'DROP'

df['severity_score'] = df.apply(get_score, axis=1)
df_clean = df[~df['severity_score'].astype(str).str.contains('DROP')].copy()
df_clean['target_class'] = df_clean['severity_score'].apply(lambda x: 0 if x == 0 else 1)

output_csv = 'master_dataset_dyslexia.csv'
df_clean.to_csv(output_csv, index=False)

print(f"SUCCESS! Disimpan ke '{output_csv}' sebanyak {len(df_clean)} file gambar bersih.")
df_clean.sample(5)

Membaca seluruh direktori...
Total gambar berserakan yang ditemukan: 208372 file.

SUCCESS! Disimpan ke 'master_dataset_dyslexia.csv' sebanyak 180726 file gambar bersih.


,image_path,file_name,split,folder_category,severity_score,target_class
155407,Gambo\Train\Normal\Normal166.png,Normal166.png,Train,Normal,0,0
72192,Gambo\Train\Corrected\1_23920.png,1_23920.png,Train,Corrected,1,1
189364,Gambo\Train\Reversal\6_6066.png,6_6066.png,Train,Reversal,6,1
13435,Gambo\Test\Corrected\4_152.png,4_152.png,Test,Corrected,4,1
27144,Gambo\Test\Normal\Normal170 (7).png,Normal170 (7).png,Test,Normal,0,0
